In [ ]:
!pip install -U peft bitsandbytes transformers accelerate

In [ ]:
!pip install -U trl

In [ ]:
!pip install -U PyMuPDF

In [ ]:
from datasets import Dataset , load_dataset

In [ ]:
# dataset = load_dataset("roneneldan/TinyStories",split='train')

In [ ]:
# dataset

In [ ]:
import fitz

In [ ]:
def extract_text_from_pdf(pdf_path):
    text_blocks = []
    with fitz.open(pdf_path) as doc:
        for page in doc:
            text = page.get_text("text").strip()
            if text:
                text_blocks.append(text)
    return text_blocks

In [ ]:
pdf_texts = extract_text_from_pdf("/content/pdfcoffee.com_leonard-s-lilly-pathophysiology-of-heart-disease-2015-pdf-free.pdf")

In [ ]:
len(pdf_texts)

In [ ]:
pdf_texts[49]

In [ ]:
import re
def split_paragraphs(pages):
    paragraphs = []
    for page_text in pages:
        chunks = re.split(r'\n\s*\n', page_text)
        for chunk in chunks:
            clean = chunk.strip()
            if len(clean) > 30:
                paragraphs.append(clean)
    return paragraphs

In [ ]:
paragraphs=split_paragraphs(pdf_texts)

In [ ]:
import re

def clean_paragraph(text):
    text = re.sub(r'\s+', ' ', text)

    text = re.sub(r'\.{2,}', ' ', text)

    text = re.sub(r'[{}~]', '', text)

    text = re.sub(r'\b\d+\b(?=\s*TABLE)', '', text)

    return text.strip()

cleaned_paragraphs = [clean_paragraph(p) for p in paragraphs]

In [ ]:
cleaned_paragraphs[10]

In [ ]:
filtered_paragraphs = [
    p for p in cleaned_paragraphs
    if not re.match(r'^TABLE\s+\d+', p.strip(), re.IGNORECASE)
]

In [ ]:
filtered_paragraphs[10]

In [ ]:
data = [{"text": p} for p in filtered_paragraphs]

In [ ]:
# data

In [ ]:
dataset = Dataset.from_list(data)

In [ ]:
dataset

In [ ]:
model_path = "Qwen/Qwen2.5-3B"

In [ ]:
from transformers import AutoTokenizer,AutoModelForCausalLM,Trainer,TrainingArguments,DataCollatorForLanguageModeling, BitsAndBytesConfig

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_path)

In [ ]:
if tokenizer.pad_token is None:
  tokenizer.pad_token = tokenizer.eos_token

In [ ]:
def tokenize_fn(examples):
  tokens = tokenizer(examples['text'],truncation=True,padding="max_length",max_length=512)
  tokens['labels']=tokens['input_ids'].copy()
  return tokens

In [ ]:
tokenized = dataset.map(tokenize_fn,batched=True,remove_columns=['text'])

In [ ]:
tokenized

In [ ]:
model = AutoModelForCausalLM.from_pretrained(model_path)

In [ ]:
training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=2,
    per_device_train_batch_size=2,
    save_steps=500,
    save_total_limit=2,
    logging_steps=50,
    learning_rate=2e-5,
    fp16=True,
    report_to="none"
 )

In [ ]:
# trainer = Trainer

In [ ]:
from peft import LoraConfig , get_peft_model , TaskType

In [ ]:
import torch

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    bnb_8bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto"
    )

In [ ]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none"
)

In [ ]:
model = get_peft_model(model,lora_config)

In [ ]:
args = TrainingArguments(
    output_dir="./tinyllama-lora",
    num_train_epochs=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_total_limit=1,
    report_to="none"
)

In [ ]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized
)

In [ ]:
trainer.train()

In [ ]:
!zip -r non_instruct_cardio.zip /content/tinyllama-lora/checkpoint-300

In [ ]:
trained_model_path = "/content/tinyllama-lora/checkpoint-300"

In [ ]:
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto"
)

trained_model = PeftModel.from_pretrained(base_model, trained_model_path)

In [ ]:
prompt = "Explain the difference between Profile B and Profile C in acute heart failure."

In [ ]:
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

In [ ]:
outputs = trained_model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

In [ ]:
outputs1 = model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

In [ ]:
print('fine_tuned_model ans:\n')
print(tokenizer.decode(outputs[0], skip_special_tokens=True),'\n')


print('base_model ans:\n')
print(tokenizer.decode(outputs1[0], skip_special_tokens=True))